# Module 3 - Grounding with a Managed Knowledge Base

In this notebook you create an Amazon Bedrock Managed Knowledge Base from AnyCompany's travel policy, and use it so that both the travel chat and the expense checks answer from the actual policy, with citations.

## Contents

1. [Introduction](#intro)
2. [Setup](#setup)
3. [The travel policy documents](#documents)
4. [Upload the documents to Amazon S3](#upload)
5. [Create the Managed Knowledge Base](#create-kb)
6. [Connect the data source and start ingestion](#ingest)
7. [What the Managed Knowledge Base handles for you](#managed)
8. [Retrieve policy passages](#retrieve)
9. [Grounded chat answers with citations](#grounded-chat)
10. [Grounded expense checks](#grounded-expense)
11. [Optional: agentic retrieval](#agentic)
12. [Try it](#try-it)
13. [Conclusion and key takeaways](#conclusion)
14. [Next steps](#next-steps)

## 1. Introduction <a id="intro"></a>

Your assistant has two gaps:

- **Travel chat** does not know AnyCompany's policy. In Module 1 it could not tell you the hotel limit in Chicago, and in Module 2 one model invented a travel portal.
- **Expense checks** use rules you typed into the prompt by hand. If the policy changes, or the hand-typed rules are incomplete, the checks are wrong.

The fix for both is **retrieval-augmented generation (RAG)**: before the model answers, find the relevant passages in AnyCompany's policy documents and give them to the model along with the question. The model then answers from the policy instead of from memory, and can cite where each answer came from.

An [Amazon Bedrock Managed Knowledge Base](https://docs.aws.amazon.com/bedrock/latest/userguide/kb-build-managed.html) does the retrieval part for you. You point it at your documents, and it handles everything needed to search them:

| You do | The Managed Knowledge Base does |
|:--|:--|
| Put documents in a data source, here Amazon S3 | Parses and splits them into passages |
| Start ingestion | Converts passages into embeddings and stores them in a managed index |
| Send a question with the `Retrieve` API | Runs hybrid search, reranks the results, returns the best passages |
| Send the passages and question to a model with Converse | |

## 2. Setup <a id="setup"></a>

Run the cell below to create the clients and set the names used in this module. The two model IDs are the ones you chose in Module 2.

In [ ]:
import glob
import json
import os
import re
import time

import boto3

# The AWS Region from your AWS configuration (or the AWS_REGION environment variable).
REGION = boto3.session.Session().region_name
ACCOUNT_ID = boto3.client("sts").get_caller_identity()["Account"]

s3 = boto3.client("s3", region_name=REGION)
iam = boto3.client("iam")
bedrock_agent = boto3.client("bedrock-agent", region_name=REGION)  # Create and manage knowledge bases
bedrock_agent_runtime = boto3.client("bedrock-agent-runtime", region_name=REGION)  # Query knowledge bases
bedrock_runtime = boto3.client("bedrock-runtime", region_name=REGION)  # Call models

# Chosen in Module 2.
CHAT_MODEL_ID = "global.anthropic.claude-sonnet-5"
EXPENSE_MODEL_ID = "openai.gpt-oss-120b-1:0"

# Names for the resources you create. The account ID keeps the bucket name globally unique.
BUCKET_NAME = f"aim317-travel-policy-{ACCOUNT_ID}-{REGION}"
ROLE_NAME = "aim317-knowledge-base-role"
KNOWLEDGE_BASE_NAME = "aim317-travel-policy"

print(f"Region:         {REGION}")
print(f"Bucket:         {BUCKET_NAME}")
print(f"Knowledge base: {KNOWLEDGE_BASE_NAME}")

## 3. The travel policy documents <a id="documents"></a>

AnyCompany's travel policy is a set of short Markdown documents in `data/policies`. In a real company these might live in SharePoint, Confluence or Google Drive, all of which a Managed Knowledge Base can connect to directly.

Run the cell below to list the documents and read the hotel policy.

In [ ]:
POLICY_FILES = sorted(glob.glob("../data/policies/*.md"))

for path in POLICY_FILES:
    print(os.path.basename(path))

print("\n" + open("../data/policies/03-hotels.md").read())

Notice that the hotel policy has a higher limit for some cities, which the hand-typed rules in Module 2 did not include. Keep that in mind for the expense checks later.

## 4. Upload the documents to Amazon S3 <a id="upload"></a>

The knowledge base reads documents from a data source. Here the data source is an Amazon S3 bucket. Run the cell below to create the bucket, if it does not exist yet, and upload the documents.

In [ ]:
try:
    s3.head_bucket(Bucket=BUCKET_NAME)
    print(f"Bucket {BUCKET_NAME} already exists")
except s3.exceptions.ClientError:
    if REGION == "us-east-1":
        s3.create_bucket(Bucket=BUCKET_NAME)
    else:
        s3.create_bucket(Bucket=BUCKET_NAME, CreateBucketConfiguration={"LocationConstraint": REGION})
    print(f"Created bucket {BUCKET_NAME}")

for path in POLICY_FILES:
    s3.upload_file(path, BUCKET_NAME, os.path.basename(path))
print(f"Uploaded {len(POLICY_FILES)} documents")

The knowledge base also needs permission to read the bucket. It uses an [IAM service role](https://docs.aws.amazon.com/bedrock/latest/userguide/kb-permissions.html) that Amazon Bedrock can assume, with read access to this bucket only.

> **Note:** At AWS events, this role already exists in your account and the cell below simply finds it. When you run the workshop in your own account, the cell creates it.

In [ ]:
def get_or_create_knowledge_base_role():
    """Return the ARN of the knowledge base service role, creating the role if needed."""
    try:
        return iam.get_role(RoleName=ROLE_NAME)["Role"]["Arn"]
    except iam.exceptions.NoSuchEntityException:
        pass

    # Who can assume the role: Amazon Bedrock, for knowledge bases in this account only.
    trust_policy = {
        "Version": "2012-10-17",
        "Statement": [{
            "Effect": "Allow",
            "Principal": {"Service": "bedrock.amazonaws.com"},
            "Action": "sts:AssumeRole",
            "Condition": {
                "StringEquals": {"aws:SourceAccount": ACCOUNT_ID},
                "ArnLike": {"aws:SourceArn": f"arn:aws:bedrock:{REGION}:{ACCOUNT_ID}:knowledge-base/*"},
            },
        }],
    }
    # What the role can do: read the policy bucket.
    permissions = {
        "Version": "2012-10-17",
        "Statement": [
            {"Effect": "Allow", "Action": "s3:ListBucket", "Resource": f"arn:aws:s3:::{BUCKET_NAME}"},
            {"Effect": "Allow", "Action": "s3:GetObject", "Resource": f"arn:aws:s3:::{BUCKET_NAME}/*"},
        ],
    }
    role = iam.create_role(RoleName=ROLE_NAME, AssumeRolePolicyDocument=json.dumps(trust_policy))
    iam.put_role_policy(RoleName=ROLE_NAME, PolicyName="read-travel-policy-bucket",
                        PolicyDocument=json.dumps(permissions))
    time.sleep(10)  # Give IAM a moment to make the new role available.
    return role["Role"]["Arn"]


ROLE_ARN = get_or_create_knowledge_base_role()
print("Service role:", ROLE_ARN)

## 5. Create the Managed Knowledge Base <a id="create-kb"></a>

Creating a Managed Knowledge Base takes one API call. The `knowledgeBaseConfiguration` sets the type to `MANAGED` and the embedding model to `MANAGED`, so Amazon Bedrock chooses and runs the embedding model for you. There is no vector database to set up.

Run the cell below to create the knowledge base, or reuse it if you already created it, and wait until it is `ACTIVE`. This usually takes less than a minute.

In [ ]:
def find_knowledge_base(name):
    """Return the ID of the knowledge base with this name, or None."""
    for kb in bedrock_agent.list_knowledge_bases(maxResults=100)["knowledgeBaseSummaries"]:
        if kb["name"] == name:
            return kb["knowledgeBaseId"]
    return None


KNOWLEDGE_BASE_ID = find_knowledge_base(KNOWLEDGE_BASE_NAME)

if KNOWLEDGE_BASE_ID is None:
    response = bedrock_agent.create_knowledge_base(
        name=KNOWLEDGE_BASE_NAME,
        description="AnyCompany travel and expense policy",
        roleArn=ROLE_ARN,
        knowledgeBaseConfiguration={
            "type": "MANAGED",
            "managedKnowledgeBaseConfiguration": {"embeddingModelType": "MANAGED"},
        },
    )
    KNOWLEDGE_BASE_ID = response["knowledgeBase"]["knowledgeBaseId"]
    print("Creating knowledge base", KNOWLEDGE_BASE_ID)

while (status := bedrock_agent.get_knowledge_base(knowledgeBaseId=KNOWLEDGE_BASE_ID)["knowledgeBase"]["status"]) == "CREATING":
    time.sleep(5)
print(f"Knowledge base {KNOWLEDGE_BASE_ID} is {status}")

## 6. Connect the data source and start ingestion <a id="ingest"></a>

The knowledge base is empty until you connect a data source and ingest it. The data source uses the managed S3 connector; `connectorParameters` names the bucket to read. Other connectors, such as SharePoint or Confluence, use the same call with different parameters.

Run the cell below to create the data source, if it does not exist yet, and start an **ingestion job**. Ingestion reads every document, splits it into passages, creates embeddings and indexes them. The call returns immediately; ingestion continues in the background.

In [ ]:
data_sources = bedrock_agent.list_data_sources(knowledgeBaseId=KNOWLEDGE_BASE_ID)["dataSourceSummaries"]

if data_sources:
    DATA_SOURCE_ID = data_sources[0]["dataSourceId"]
else:
    response = bedrock_agent.create_data_source(
        knowledgeBaseId=KNOWLEDGE_BASE_ID,
        name="travel-policy-s3",
        dataDeletionPolicy="DELETE",  # Remove indexed data when the data source is deleted.
        dataSourceConfiguration={
            "type": "MANAGED_KNOWLEDGE_BASE_CONNECTOR",
            "managedKnowledgeBaseConnectorConfiguration": {
                "connectorParameters": {
                    "type": "S3",
                    "version": "1",
                    "connectionConfiguration": {
                        "bucketName": BUCKET_NAME,
                        "bucketOwnerAccountId": ACCOUNT_ID,
                    },
                    "deletionProtectionConfiguration": {"enableDeletionProtection": False},
                }
            },
        },
    )
    DATA_SOURCE_ID = response["dataSource"]["dataSourceId"]
    while bedrock_agent.get_data_source(knowledgeBaseId=KNOWLEDGE_BASE_ID, dataSourceId=DATA_SOURCE_ID)["dataSource"]["status"] == "CREATING":
        time.sleep(5)

ingestion_job = bedrock_agent.start_ingestion_job(knowledgeBaseId=KNOWLEDGE_BASE_ID, dataSourceId=DATA_SOURCE_ID)
INGESTION_JOB_ID = ingestion_job["ingestionJob"]["ingestionJobId"]
print(f"Data source {DATA_SOURCE_ID}: ingestion job {INGESTION_JOB_ID} started")

## 7. What the Managed Knowledge Base handles for you <a id="managed"></a>

Ingestion takes about two to three minutes for these documents. While it runs, here is what is happening, and what you would otherwise build and operate yourself:

| Step | What happens | Without a managed knowledge base |
|:--|:--|:--|
| **Parsing** | Text and tables are extracted from each document (PDF, Word, HTML, Markdown and more) | Choose and run a parser for each file type |
| **Chunking** | Documents are split into passages small enough to search and to fit in a prompt | Choose a chunk size and overlap, and tune them |
| **Embedding** | Each passage becomes a vector that represents its meaning | Choose an embedding model, call it for every passage, handle limits |
| **Indexing** | Vectors and text are stored in a managed index | Provision, size and pay for a vector database |
| **Hybrid search** | Each query is matched both by meaning and by keywords | Combine semantic and keyword search yourself |
| **Reranking** | The best candidates are rescored for relevance before being returned | Add and call a reranking model |

Keyword matching matters for questions like "What is the limit in Zurich?": a city name is an exact term, and meaning alone can miss it.

Run the cell below to wait for ingestion to finish. It checks every 15 seconds.

> **Note:** At AWS events, if ingestion has not finished after 8 minutes, the cell switches to a knowledge base that was prepared in your account in advance, so you can continue.

In [ ]:
FALLBACK_KNOWLEDGE_BASE_NAME = "aim317-travel-policy-prebuilt"
deadline = time.time() + 8 * 60

while True:
    job = bedrock_agent.get_ingestion_job(
        knowledgeBaseId=KNOWLEDGE_BASE_ID, dataSourceId=DATA_SOURCE_ID, ingestionJobId=INGESTION_JOB_ID
    )["ingestionJob"]
    if job["status"] not in ("STARTING", "IN_PROGRESS"):
        break
    if time.time() > deadline and (fallback_id := find_knowledge_base(FALLBACK_KNOWLEDGE_BASE_NAME)):
        KNOWLEDGE_BASE_ID = fallback_id
        print(f"Ingestion is taking longer than expected. Using the prepared knowledge base {fallback_id}.")
        break
    print(f"{time.strftime('%H:%M:%S')}  {job['status']}")
    time.sleep(15)

print(f"Ingestion {job['status']}")
stats = job.get("statistics", {})
print(f"Documents indexed: {stats.get('numberOfNewDocumentsIndexed', 0) + stats.get('numberOfModifiedDocumentsIndexed', 0)}, "
      f"failed: {stats.get('numberOfDocumentsFailed', 0)}")

## 8. Retrieve policy passages <a id="retrieve"></a>

With ingestion complete, you can search the policy. The [`Retrieve`](https://docs.aws.amazon.com/bedrock/latest/APIReference/API_agent-runtime_Retrieve.html) API takes a question and returns the most relevant passages, each with its source document and a relevance score. For a Managed Knowledge Base, you set the number of results in `managedSearchConfiguration`.

Run the cell below to search for the question your assistant could not answer in Module 1.

In [ ]:
def retrieve(query, number_of_results=5):
    """Return the most relevant policy passages for a query."""
    response = bedrock_agent_runtime.retrieve(
        knowledgeBaseId=KNOWLEDGE_BASE_ID,
        retrievalQuery={"text": query},
        retrievalConfiguration={"managedSearchConfiguration": {"numberOfResults": number_of_results}},
    )
    return [
        {
            "text": result["content"]["text"],
            "source": result["location"]["s3Location"]["uri"].split("/")[-1],
            "score": result.get("score", 0),
        }
        for result in response["retrievalResults"]
    ]


for passage in retrieve("What is AnyCompany's nightly hotel limit in Chicago?", number_of_results=3):
    print(f"{passage['score']:.2f}  {passage['source']}")
    print("      " + passage["text"][:150].replace("\n", " ") + "...\n")

The hotel policy comes first, with a much higher score than the others. These documents are short, so each one fits in a single passage; longer documents are split into many passages, and only the relevant ones are returned.

## 9. Grounded chat answers with citations <a id="grounded-chat"></a>

Now combine retrieval with the chat model. The `ask_travel_assistant` function below:

1. Retrieves the most relevant policy passages for the question.
2. Numbers them and puts them in the prompt, inside `<policy>` tags.
3. Tells the model to answer only from those passages, and to cite them by number.
4. Returns the answer and the list of sources, so you can show employees where each answer came from.

> **Note:** Amazon Bedrock also has a `RetrieveAndGenerate` API that does both steps in one call, but it does not support Managed Knowledge Bases. Calling `Retrieve` and Converse yourself also lets you use any model and control the prompt.

In [ ]:
def get_text(response):
    """Return the text blocks of a Converse response, joined together."""
    return "".join(block["text"] for block in response["output"]["message"]["content"] if "text" in block)


def format_passages(passages):
    """Number the passages so the model can cite them."""
    return "\n\n".join(f"[{i}] (source: {p['source']})\n{p['text']}" for i, p in enumerate(passages, start=1))


GROUNDED_CHAT_PROMPT = (
    "You are the travel assistant for AnyCompany, helping employees plan and manage business "
    "travel. Answer using only the AnyCompany policy passages inside the <policy> tags. Cite the "
    "passages you use by number, like [1]. If the passages do not answer the question, say so and "
    "suggest contacting the AnyCompany Travel Desk. Be friendly and concise: three sentences or "
    "fewer unless the employee asks for detail."
)


def ask_travel_assistant(question):
    """Answer a travel question from the policy, and return the answer with its sources."""
    passages = retrieve(question)
    response = bedrock_runtime.converse(
        modelId=CHAT_MODEL_ID,
        system=[{"text": GROUNDED_CHAT_PROMPT}],
        messages=[{
            "role": "user",
            "content": [{"text": f"<policy>\n{format_passages(passages)}\n</policy>\n\nQuestion: {question}"}],
        }],
        inferenceConfig={"maxTokens": 2000},
    )
    return get_text(response), passages


def show_answer(question):
    answer, passages = ask_travel_assistant(question)
    print(f"Q: {question}\n\n{answer}\n")
    cited = sorted({int(n) for n in re.findall(r"\[(\d+)\]", answer)})
    for n in cited:
        if 1 <= n <= len(passages):
            print(f"  [{n}] {passages[n - 1]['source']}")


show_answer("What is AnyCompany's nightly hotel limit in Chicago?")

The assistant now answers with AnyCompany's actual limit and cites the hotel policy. Run the cell below to ask the question from Module 2, where one model invented a travel portal.

In [ ]:
show_answer(
    "My hotel in Chicago is overbooked and has no room for me tonight. "
    "I have a client meeting at 9am tomorrow. What should I do right now?"
)

This time the answer follows AnyCompany's trip disruption policy: the steps to take, the Travel Desk number, and the 25% exception for that night.

Finally, ask something the policy does not cover. A grounded assistant should say it does not know, rather than guess.

In [ ]:
show_answer("Can I bring my dog on a business trip?")

## 10. Grounded expense checks <a id="grounded-expense"></a>

Now apply the same idea to the expense checks. Instead of rules typed into the prompt, each check retrieves the relevant policy passages for that report.

A report can mix hotels, meals and flights, and one search for all of them at once can miss one of the policies. So `check_expense` runs **one search per expense type** in the report, and combines the results. Retrieving well is as much a part of RAG as generating well.

Run the cell below to load the expense reports from Module 2 and define the grounded check.

In [ ]:
with open("../data/expense_reports.json") as f:
    EXPENSE_REPORTS = json.load(f)


def format_report(report):
    """Turn an expense report into the text the model reads."""
    lines = [f"Expense report {report['id']} - {report['employee']} - {report['trip']}"]
    for item in report["items"]:
        lines.append(f"- {item['type']}: {item['description']}: ${item['amount']:.2f}")
    return "\n".join(lines)


GROUNDED_EXPENSE_PROMPT = (
    "You check AnyCompany expense reports against the AnyCompany policy passages inside the "
    "<policy> tags. Use only those passages. Reply with exactly two lines: first line APPROVED "
    "or FLAGGED, second line a one-sentence reason that cites the passage number, like [1]."
)


def get_verdict(text):
    """Find the first APPROVED or FLAGGED in a reply."""
    match = re.search(r"\b(APPROVED|FLAGGED)\b", text)
    return match.group(1) if match else "UNKNOWN"


def check_expense(report):
    """Check one expense report against the retrieved policy, and return the verdict and reason."""
    passages = []
    for item_type in sorted({item["type"] for item in report["items"]}):
        for passage in retrieve(f"AnyCompany {item_type} policy limits for a trip to {report['trip']}", 2):
            if passage["source"] not in [p["source"] for p in passages]:
                passages.append(passage)
    response = bedrock_runtime.converse(
        modelId=EXPENSE_MODEL_ID,
        system=[{"text": GROUNDED_EXPENSE_PROMPT}],
        messages=[{
            "role": "user",
            "content": [{"text": f"<policy>\n{format_passages(passages)}\n</policy>\n\n{format_report(report)}"}],
        }],
        inferenceConfig={"maxTokens": 2000},
    )
    text = get_text(response).strip()
    return get_verdict(text), text.splitlines()[-1]

Each report in `data/expense_reports.json` has two expected verdicts: `expected`, from the hand-typed rules in Module 2, and `expected_with_policy`, from the full policy. Run the cell below to check every report against the policy and compare.

In [ ]:
correct = 0
for report in EXPENSE_REPORTS:
    verdict, reason = check_expense(report)
    correct += verdict == report["expected_with_policy"]
    changed = "  <- changed from Module 2" if report["expected"] != report["expected_with_policy"] else ""
    print(f"{report['id']}  {verdict:<9} expected {report['expected_with_policy']:<9}{changed}")
    print(f"         {reason}\n")

print(f"{correct}/{len(EXPENSE_REPORTS)} match the policy")

Look at **EXP-002**, Marcus's three nights in New York at $320 a night. The hand-typed rule in Module 2 ($250 per night everywhere) flagged it. The actual policy allows $325 a night in New York, so it is approved, and the reason cites the hotel policy.

The hand-typed rules were not wrong about what they said; they were incomplete. Grounding the check in the source documents means it stays correct when the policy has details you did not think to copy, and when the policy changes.

## 11. Optional: agentic retrieval <a id="agentic"></a>

`Retrieve` runs one search for one query. Some questions need several searches, for example a question about a hotel, meals and a flight at once. [Agentic retrieval](https://docs.aws.amazon.com/bedrock/latest/APIReference/API_agent-runtime_AgenticRetrieveStream.html) uses a model to plan the searches, runs them, and can generate an answer from the results. It takes longer and costs more per call than `Retrieve`, so use it for complex questions.

Run the cell below to send a multi-part question and watch the steps as they stream back.

In [ ]:
response = bedrock_agent_runtime.agentic_retrieve_stream(
    messages=[{
        "role": "user",
        "content": {"text": "I'm flying from New York to London for 3 nights. Which flight class, "
                            "hotel limit and daily meal limit apply?"},
    }],
    retrievers=[{
        "configuration": {"knowledgeBase": {"knowledgeBaseId": KNOWLEDGE_BASE_ID}},
        "description": "AnyCompany travel and expense policy",
    }],
    agenticRetrieveConfiguration={"foundationModelType": "MANAGED"},
)

for event in response["stream"]:
    if "traceEvent" in event:
        attributes = event["traceEvent"]["attributes"]
        print(f"[{attributes['step']} {attributes['status']}]")
    elif "result" in event:
        result = event["result"]
        print("\nSources:", sorted({r["metadata"]["_source_uri"].split("/")[-1] for r in result["results"]}))
        if result.get("generatedResponse"):
            print("\nAnswer:", result["generatedResponse"]["answer"])

## 12. Try it <a id="try-it"></a>

Policies change. Update one and see the assistant follow.

1. Open `data/policies/03-hotels.md` and add Chicago to the high-cost cities, at $325 a night.
2. Run the upload cell in section 4, then the ingestion cells in sections 6 and 7 again. The knowledge base re-indexes only the changed document.
3. Run the cell below. The answer should now reflect the new limit, without any change to your code or prompts.

In [ ]:
show_answer("What is AnyCompany's nightly hotel limit in Chicago?")

> **Note:** If you make this change, undo it and re-ingest before Module 4, so your results match the rest of the workshop.

## 13. Conclusion and key takeaways <a id="conclusion"></a>

- **RAG** grounds a model's answers in your own documents: retrieve the relevant passages, then generate from them.
- A **Managed Knowledge Base** handles parsing, chunking, embeddings, indexing, hybrid search and reranking. You create it with one call, connect a data source, and ingest.
- **`Retrieve` plus Converse** gives you grounded answers from any model, with citations you can show to users.
- Grounding fixed both tasks: chat answers follow the real policy and admit what it does not cover, and expense checks use the full policy instead of hand-typed rules.
- When the policy changes, you **re-ingest the documents**; the code and prompts stay the same.

## 14. Next steps <a id="next-steps"></a>

Your assistant now answers from AnyCompany's policy. In [Module 4 - Guardrails](04_guardrails.ipynb), you keep it safe and on-policy: redacting personal data such as passport numbers, refusing to help book outside policy, and checking that answers are supported by the passages you retrieved.

The knowledge base you created stays in your account for the rest of the workshop. The [Cleanup](99_cleanup.ipynb) notebook deletes it at the end.